# Lesson 04: 다양한 형태의 물체 - 상자, 구, 실린더

## 학습 목표
1. `ChBodyEasy*` 간편 생성법 배우기
2. 밀도(density)로 질량/관성 자동 계산
3. 세 가지 기본 형태: 구, 상자, 실린더

## 새로 배우는 Chrono API
| 클래스 | 파라미터 | 자동 생성되는 것 |
|---|---|---|
| `ChBodyEasySphere(r, density, vis, collide, mat)` | 반지름, 밀도 | 질량, 관성, 충돌형태, 시각형태 |
| `ChBodyEasyBox(x,y,z, density, vis, collide, mat)` | 크기, 밀도 | 위와 동일 |
| `ChBodyEasyCylinder(axis, r, h, density, vis, collide, mat)` | 축, 반지름, 높이, 밀도 | 위와 동일 |

> `ChBodyEasy*`는 Lesson 02~03에서 수동으로 했던 작업(충돌 형태 + 시각 형태 + 관성 계산)을 한 줄로 해결합니다.

In [ ]:
import pychrono as chrono

sys = chrono.ChSystemNSC()
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET)

material = chrono.ChContactMaterialNSC()
material.SetFriction(0.5)
material.SetRestitution(0.3)

## ChBody 수동 생성 vs ChBodyEasy 비교

```python
# 수동 (Lesson 02~03 방식) — 6줄
ball = chrono.ChBody()
ball.SetMass(1.0)
ball.SetInertiaXX(chrono.ChVector3d(0.036, 0.036, 0.036))
ball.EnableCollision(True)
ball.AddCollisionShape(chrono.ChCollisionShapeSphere(material, 0.3))
ball.AddVisualShape(chrono.ChVisualShapeSphere(0.3))

# 간편 (이번 레슨) — 1줄
ball = chrono.ChBodyEasySphere(0.3, 1000, True, True, material)
```

## 구(Sphere) — 철(steel), 밀도 7800 kg/m³

In [ ]:
# ChBodyEasySphere(반지름, 밀도, 시각화, 충돌, 재질)
sphere = chrono.ChBodyEasySphere(0.4, 7800, True, True, material)
sphere.SetPos(chrono.ChVector3d(-3, 6, 0))
sphere.GetVisualShape(0).SetColor(chrono.ChColor(0.9, 0.2, 0.2))  # 빨강
sys.AddBody(sphere)

# 질량 = 밀도 × 체적 = 7800 × (4/3)π(0.4)³
import math
expected_mass = 7800 * (4/3) * math.pi * 0.4**3
print(f"구: 반지름=0.4m, 밀도=7800 kg/m³ (철)")
print(f"  자동 계산 질량: {sphere.GetMass():.2f} kg")
print(f"  수식 검증:      {expected_mass:.2f} kg")

## 상자(Box) — 나무(wood), 밀도 600 kg/m³

In [ ]:
# ChBodyEasyBox(x크기, y크기, z크기, 밀도, 시각화, 충돌, 재질)
box = chrono.ChBodyEasyBox(0.8, 0.8, 0.8, 600, True, True, material)
box.SetPos(chrono.ChVector3d(0, 5, 0))
box.GetVisualShape(0).SetColor(chrono.ChColor(0.2, 0.6, 0.9))  # 파랑
sys.AddBody(box)

# 질량 = 밀도 × 체적 = 600 × 0.8³
expected_mass = 600 * 0.8**3
print(f"상자: 0.8×0.8×0.8m, 밀도=600 kg/m³ (나무)")
print(f"  자동 계산 질량: {box.GetMass():.2f} kg")
print(f"  수식 검증:      {expected_mass:.2f} kg")

## 실린더(Cylinder) — 알루미늄, 밀도 2700 kg/m³

In [ ]:
# ChBodyEasyCylinder(축방향, 반지름, 높이, 밀도, 시각화, 충돌, 재질)
# ChAxis_Y = Y축 방향으로 세운 실린더
cylinder = chrono.ChBodyEasyCylinder(
    chrono.ChAxis_Y,    # 실린더 축 방향
    0.3,                # 반지름
    1.0,                # 높이
    2700,               # 밀도 (알루미늄)
    True,               # 시각화
    True,               # 충돌
    material
)
cylinder.SetPos(chrono.ChVector3d(3, 7, 0))
cylinder.GetVisualShape(0).SetColor(chrono.ChColor(0.9, 0.8, 0.1))  # 노랑
sys.AddBody(cylinder)

expected_mass = 2700 * math.pi * 0.3**2 * 1.0
print(f"실린더: 반지름=0.3m, 높이=1.0m, 밀도=2700 kg/m³ (알루미늄)")
print(f"  자동 계산 질량: {cylinder.GetMass():.2f} kg")
print(f"  수식 검증:      {expected_mass:.2f} kg")

## 반복 생성 — for 루프로 대량 배치

In [ ]:
small_balls = []
for i in range(5):
    sb = chrono.ChBodyEasySphere(0.2, 3000, True, True, material)
    sb.SetPos(chrono.ChVector3d(-2 + i, 10 + i, 0.5 * i))
    sb.GetVisualShape(0).SetColor(
        chrono.ChColor(0.6 + i * 0.08, 0.3, 0.8 - i * 0.1)
    )
    sys.AddBody(sb)
    small_balls.append(sb)

print(f"작은 구 {len(small_balls)}개 추가")
print(f"총 물체 수: {len(sys.GetBodies())} (바닥 미포함 시 -1)")

## 시뮬레이션 실행 (시각화 없이)

In [ ]:
# 바닥 추가 (시각화 없이도 충돌을 위해 필요)
floor = chrono.ChBody()
floor.SetPos(chrono.ChVector3d(0, -1, 0))
floor.SetFixed(True)
floor.EnableCollision(True)
floor.AddCollisionShape(chrono.ChCollisionShapeBox(material, 30, 2, 30))
sys.AddBody(floor)

# 5초 시뮬레이션
while sys.GetChTime() < 5.0:
    sys.DoStepDynamics(0.005)

print(f"시뮬레이션 {sys.GetChTime():.2f}초 완료")
print(f"  구(철):       y = {sphere.GetPos().y:.3f} m, 질량 = {sphere.GetMass():.1f} kg")
print(f"  상자(나무):    y = {box.GetPos().y:.3f} m, 질량 = {box.GetMass():.1f} kg")
print(f"  실린더(알루미늄): y = {cylinder.GetPos().y:.3f} m, 질량 = {cylinder.GetMass():.1f} kg")

## 정리: 자주 쓰는 밀도 참고값

| 재료 | 밀도 (kg/m³) |
|---|---|
| 공기 | 1.2 |
| 나무 (소나무) | 500~600 |
| 물 | 1000 |
| 콘크리트 | 2400 |
| 알루미늄 | 2700 |
| 철/강철 | 7800 |
| 구리 | 8900 |
| 납 | 11340 |